# Estimación de Carbono Aéreo — Reserva de Biósfera Pereyra Iraola
## Pipeline: GEDI L2A + Sentinel-2 L2A + Random Forest

**Autora:** Darío Nicolás Sánchez Leguizamón  
**Institución:** Universidad Nacional Arturo Jauretche (UNAJ), Argentina  
**Contacto:** sdarionicolas@gmail.com  
**Paper:** *Estimación del carbono aéreo en la Reserva de Biósfera Pereyra Iraola mediante integración de datos GEDI, Sentinel-2 e inteligencia artificial*  
**Congreso:** CAI 2026 — 55 JAIIO, La Plata, Argentina  
**Repositorio:** https://github.com/sdarionicolas-boop/carbono-pereyra-gedi

---

### Descripción
Pipeline computacional modular y reproducible para la estimación y cartografía de carbono aéreo
en la Reserva de Biósfera Pereyra Iraola (~10.248 ha, Buenos Aires, Argentina).

**Tecnologías:** Google Earth Engine · Google Colab · scikit-learn · rasterio · geopandas

**Métricas del modelo:** R² = 0.71 · RMSE = 7.62 tC/ha · MAE = 5.14 tC/ha

---

### Módulos del pipeline

| Módulo | Descripción |
|--------|-------------|
| 0 | Configuración y autenticación |
| 1 | Adquisición y exportación GEDI L2A desde GEE |
| 2 | Adquisición y exportación Sentinel-2 L2A desde GEE |
| 3 | Estimación de biomasa y carbono aéreo (alometría) |
| 4 | Random Forest altura → carbono (validación intermedia) |
| 5 | Random Forest multibanda Sentinel-2 → carbono (modelo principal) |
| 6 | Validación visual: scatter plot, mapa de carbono, residuos |
| 7 | Análisis estratificado del error por clase de altura |
| 8 | Importancia de variables espectrales |

---

### Requisitos previos
1. Cuenta de Google con acceso a Google Earth Engine
2. Proyecto GEE activo (reemplazá `TU_PROYECTO_GEE` en la celda de configuración)
3. Carpeta `PEREYRA/` en tu Google Drive para almacenar los GeoTIFFs exportados


## Módulo 0 — Configuración y autenticación

In [ ]:
# ============================================================
# MÓDULO 0 — Instalación de dependencias
# ============================================================
# Ejecutar solo la primera vez en el entorno Colab

!pip install geemap geopandas rasterio matplotlib pandas scikit-learn --quiet


In [ ]:
# ============================================================
# PARÁMETROS GLOBALES — Editá solo esta sección
# ============================================================

# ── GEE ──────────────────────────────────────────────────────
GEE_PROJECT = 'TU_PROYECTO_GEE'          # Reemplazá con tu proyecto GEE

# ── Área de estudio ───────────────────────────────────────────
# Coordenadas del bounding box de la Reserva Pereyra Iraola (WGS84)
COORDS_RESERVA = [
    [-58.2330, -34.8100],   # Noroeste
    [-58.2330, -34.8900],   # Suroeste
    [-58.0430, -34.8900],   # Sureste
    [-58.0430, -34.8100],   # Noreste
    [-58.2330, -34.8100],   # Cierre del polígono
]

# ── Google Drive ──────────────────────────────────────────────
DRIVE_FOLDER = 'PEREYRA'                  # Carpeta en tu Google Drive

# ── Sentinel-2 ───────────────────────────────────────────────
S2_YEAR      = '2023'                     # Año del mosaico anual
S2_CLOUD_PCT = 10                         # Umbral máximo de nubosidad (%)

# ── GEDI ─────────────────────────────────────────────────────
GEDI_SENSITIVITY = 0.95                   # Umbral mínimo de sensibilidad
GEDI_ELEV_DIFF   = 50                     # Diferencia altimétrica GEDI-SRTM máx (m)

# ── Modelo Random Forest ─────────────────────────────────────
RF_N_ESTIMATORS = 300
RF_MAX_DEPTH    = 20
RF_TEST_SIZE    = 0.30
RF_RANDOM_STATE = 42

# ── Alometría ────────────────────────────────────────────────
WOOD_DENSITY = 0.6                        # Densidad de madera ρ (g/cm³)
CARBON_FACTOR = 0.47                      # Factor de conversión biomasa → carbono (Winrock, 2018)


In [ ]:
# ============================================================
# MÓDULO 0 — Importaciones y autenticación
# ============================================================

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import os
from rasterio.enums import Resampling
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Autenticación GEE ────────────────────────────────────────
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

# ── Montar Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Ruta de salida ────────────────────────────────────────────
OUTPUT_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Carpeta de salida: {OUTPUT_DIR}')


## Módulo 1 — Adquisición y exportación GEDI L2A desde GEE

In [ ]:
# ============================================================
# MÓDULO 1 — GEDI L2A: ingesta y filtrado
# ============================================================
# Protocolo de filtrado: Saatchi y Favrichon (2023); Lei et al. (2024)
#   quality_flag = 1, degrade_flag = 0, sensibilidad > GEDI_SENSITIVITY,
#   adquisición nocturna, |GEDI_elev - SRTM_elev| < GEDI_ELEV_DIFF,
#   solo power beams.

roi = ee.Geometry.Polygon([COORDS_RESERVA])

gedi = (
    ee.ImageCollection('LARSE/GEDI/GEDI02_A_002_MONTHLY')
    .filterBounds(roi)
    .map(lambda img: img.clip(roi))
)

def filtrar_gedi(img):
    quality  = img.select('quality_flag').eq(1)
    degrade  = img.select('degrade_flag').eq(0)
    sensit   = img.select('sensitivity').gte(GEDI_SENSITIVITY)
    elev_ok  = img.select('elev_lowestmode').subtract(
                    ee.Image('USGS/SRTMGL1_003').select('elevation')
               ).abs().lte(GEDI_ELEV_DIFF)
    mask = quality.And(degrade).And(sensit).And(elev_ok)
    return img.updateMask(mask)

gedi_filtrado = gedi.map(filtrar_gedi)

# Métrica RH95: percentil 95 de energía de retorno (altura dominante del dosel)
rh95 = gedi_filtrado.select('rh95').median().clip(roi)

# Exportar a Google Drive
task_gedi = ee.batch.Export.image.toDrive(
    image=rh95.toFloat(),
    description='GEDI_RH95_Pereyra',
    folder=DRIVE_FOLDER,
    fileNamePrefix='GEDI_RH95_Pereyra',
    region=roi,
    scale=25,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_gedi.start()
print('🚀 Exportación GEDI iniciada. Verificá el panel Tasks en GEE.')
print('   Archivo de salida: GEDI_RH95_Pereyra.tif')


## Módulo 2 — Adquisición y exportación Sentinel-2 L2A desde GEE

In [ ]:
# ============================================================
# MÓDULO 2 — Sentinel-2 L2A: mosaico anual y stack multibanda
# ============================================================
# Colección: COPERNICUS/S2_SR_HARMONIZED
# Filtro de nubosidad: banda QA60 + umbral S2_CLOUD_PCT
# Mosaico: mediana espectral anual (píxel a píxel)
# Bandas exportadas: B2, B3, B4, B8, B11, B12, NDVI, NDWI

roi = ee.Geometry.Polygon([COORDS_RESERVA])

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(f'{S2_YEAR}-01-01', f'{S2_YEAR}-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_PCT))
)

mosaico = s2.median().clip(roi)

# Bandas espectrales base
bands  = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
mosaic = mosaico.select(bands)

# Índices espectrales derivados
ndvi = mosaic.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = mosaic.normalizedDifference(['B3', 'B11']).rename('NDWI')

# Stack completo (8 capas)
stack = mosaic.addBands([ndvi, ndwi])

# Visualización rápida
Map = geemap.Map()
Map.centerObject(roi, 12)
Map.addLayer(stack, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'RGB')
Map.addLayer(ndvi, {'min': 0, 'max': 1, 'palette': ['white', 'green']}, 'NDVI')
Map



In [ ]:
# ── Exportar stack Sentinel-2 a Google Drive ─────────────────
task_s2 = ee.batch.Export.image.toDrive(
    image=stack.select(['B2','B3','B4','B8','B11','B12','NDVI','NDWI']).toFloat(),
    description='Sentinel2_Pereyra_Stack',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Sentinel2_Pereyra_Stack',
    region=roi,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_s2.start()
print('🚀 Exportación Sentinel-2 iniciada.')
print('   Archivo de salida: Sentinel2_Pereyra_Stack.tif')
print('   ⚠️  Esperá a que finalice antes de continuar con el Módulo 3.')


## Módulo 3 — Estimación de biomasa y carbono aéreo (alometría)

Convierte la altura del dosel derivada de GEDI (RH95) en carbono aéreo (tC/ha)
en tres pasos:

1. **Modelo H:D** — regresión potencial altura → diámetro
2. **Ecuación de Chave et al. (2014):** AGB = 0.0673 × (ρ × D² × H)⁰·⁹⁷⁶
3. **Conversión biomasa → carbono:** C = 0.47 × AGB (Winrock International, 2018)


In [ ]:
# ============================================================
# MÓDULO 3 — Estimación alométrica de carbono aéreo
# ============================================================

altura_tif   = OUTPUT_DIR + 'GEDI_RH95_Pereyra.tif'
carbono_tif  = OUTPUT_DIR + 'Carbono_Pereyra.tif'

def rh95_a_carbon(h, rho=WOOD_DENSITY, cf=CARBON_FACTOR):
    """
    Estima carbono aéreo (tC/ha) a partir de altura RH95 (m).

    Parámetros
    ----------
    h   : array de alturas del dosel (m)
    rho : densidad de madera (g/cm³), por defecto WOOD_DENSITY
    cf  : factor de conversión biomasa → carbono, por defecto CARBON_FACTOR

    Retorna
    -------
    array de carbono aéreo (tC/ha)
    """
    # Modelo H:D empírico (regresión potencial)
    D = 2.5 * (h ** 0.63)                      # diámetro estimado (cm)
    # Ecuación alométrica Chave et al. (2014)
    agb = 0.0673 * (rho * D**2 * h) ** 0.976   # t/ha
    return agb * cf                             # tC/ha

# Leer GEDI y aplicar alometría
with rasterio.open(altura_tif) as src:
    scale_factor = 1   # sin remuestreo adicional
    alt = src.read(1).astype(float)
    meta = src.meta.copy()

# Aplicar función
carbon = rh95_a_carbon(alt)
carbon[np.isnan(alt)] = np.nan

# Guardar GeoTIFF de carbono
meta.update(dtype='float32', count=1)
with rasterio.open(carbono_tif, 'w', **meta) as dst:
    dst.write(carbon.astype('float32'), 1)

print(f'✅ Mapa de carbono guardado: {carbono_tif}')
print(f'   Media: {np.nanmean(carbon):.2f} tC/ha')
print(f'   Máximo: {np.nanmax(carbon):.2f} tC/ha')
print(f'   Stock total aprox.: {np.nansum(carbon) * 0.01:.2f} ktoneladas C')


## Módulo 4 — Random Forest altura → carbono (validación intermedia)

In [ ]:
# ============================================================
# MÓDULO 4 — Random Forest univariado (altura → carbono)
# ============================================================
# Modelo de referencia para comparar con el multibanda (Módulo 5)

altura_class_tif = OUTPUT_DIR + 'Altura_Dosel_Pereyra_Class.tif'
carbono_tif      = OUTPUT_DIR + 'Carbono_Pereyra.tif'

# Cargar rasters
with rasterio.open(carbono_tif) as src_c:
    target_h, target_w = src_c.height, src_c.width
    carbono = src_c.read(1).astype(float)

with rasterio.open(altura_class_tif) as src_a:
    altura = src_a.read(
        1,
        out_shape=(1, target_h, target_w),
        resampling=Resampling.nearest
    ).astype(float)

# Dataset
mask = (~np.isnan(altura)) & (~np.isnan(carbono)) & (carbono > 0)
X_alt = altura[mask].reshape(-1, 1)
y     = carbono[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_alt, y, test_size=RF_TEST_SIZE, random_state=RF_RANDOM_STATE
)

# Entrenamiento
rf_alt = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=RF_RANDOM_STATE,
    n_jobs=-1
)
rf_alt.fit(X_train, y_train)
y_pred_alt = rf_alt.predict(X_test)

r2   = r2_score(y_test, y_pred_alt)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_alt))
mae  = mean_absolute_error(y_test, y_pred_alt)

print('📊 RF altura → carbono (modelo intermedio)')
print(f'   R²:   {r2:.3f}')
print(f'   RMSE: {rmse:.2f} tC/ha')
print(f'   MAE:  {mae:.2f} tC/ha')
print()
print('ℹ️  El modelo principal (multibanda) está en el Módulo 5.')


## Módulo 5 — Random Forest multibanda Sentinel-2 → carbono (modelo principal)

**Este es el modelo central del paper.**

- **Variables predictoras (X):** B2, B3, B4, B8, B11, B12, NDVI, NDWI (8 capas)
- **Variable objetivo (y):** carbono aéreo (tC/ha)
- **Muestras:** 873.668 píxeles válidos
- **Resultados:** R² = 0.71 · RMSE = 7.62 tC/ha · MAE = 5.14 tC/ha


In [ ]:
# ============================================================
# MÓDULO 5 — Random Forest multibanda Sentinel-2 → carbono
# ============================================================

stack_tif    = OUTPUT_DIR + 'Sentinel2_Pereyra_Stack.tif'
carbono_tif  = OUTPUT_DIR + 'Carbono_Pereyra.tif'
predicho_tif = OUTPUT_DIR + 'Carbono_Predicho_RF.tif'

# ── 1. Remuestrear carbono a resolución Sentinel-2 (10 m) ────
carbono_10m_tif = OUTPUT_DIR + 'Carbono_Pereyra_10m.tif'

with rasterio.open(stack_tif) as ref:
    profile_ref = ref.profile.copy()
    w_ref, h_ref = ref.width, ref.height

with rasterio.open(carbono_tif) as src:
    data_c = src.read(
        out_shape=(1, h_ref, w_ref),
        resampling=Resampling.bilinear
    )
    profile_ref.update(count=1, dtype='float32')

with rasterio.open(carbono_10m_tif, 'w', **profile_ref) as dst:
    dst.write(data_c.astype('float32'))

print(f'✅ Carbono remuestreado a 10 m: {carbono_10m_tif}')

# ── 2. Cargar stack y carbono ─────────────────────────────────
with rasterio.open(stack_tif) as src:
    stack = src.read().astype(float)
    meta_stack = src.meta.copy()

with rasterio.open(carbono_10m_tif) as src:
    carbono = src.read(1).astype(float)

# ── 3. Máscara de píxeles válidos ─────────────────────────────
mask = (~np.isnan(carbono)) & (carbono > 0)
for b in stack:
    mask &= ~np.isnan(b)

X = stack[:, mask].T   # shape: (n_muestras, 8_bandas)
y = carbono[mask]

print(f'📊 Muestras válidas: {len(y):,}')

# ── 4. División train / test ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=RF_TEST_SIZE, random_state=RF_RANDOM_STATE
)

# ── 5. Entrenar modelo ────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RF_RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('✅ Modelo entrenado.')

# ── 6. Validar ────────────────────────────────────────────────
y_pred = rf.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print()
print('📊 MÉTRICAS DE VALIDACIÓN (Módulo 5 — modelo principal)')
print(f'   R²:   {r2:.3f}')
print(f'   RMSE: {rmse:.2f} tC/ha')
print(f'   MAE:  {mae:.2f} tC/ha')

# ── 7. Mapa predicho completo ─────────────────────────────────
carbono_pred_full = np.full(carbono.shape, np.nan)
X_full = stack[:, mask].T
carbono_pred_full[mask] = rf.predict(X_full)

meta_out = meta_stack.copy()
meta_out.update(count=1, dtype='float32')
with rasterio.open(predicho_tif, 'w', **meta_out) as dst:
    dst.write(carbono_pred_full.astype('float32'), 1)

print(f'✅ Mapa predicho guardado: {predicho_tif}')


## Módulo 6 — Validación visual: scatter plot y mapa de carbono

In [ ]:
# ============================================================
# MÓDULO 6 — Scatter plot: carbono observado vs. predicho
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, s=15, alpha=0.3, color='steelblue', label='Predicciones')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        'r--', lw=2.5, label='1:1')

ax.text(0.05, 0.95,
        f'R² = {r2:.3f}\nRMSE = {rmse:.2f} tC/ha\nMAE = {mae:.2f} tC/ha',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Carbono observado (tC/ha)', fontsize=12, fontweight='bold')
ax.set_ylabel('Carbono predicho (tC/ha)', fontsize=12, fontweight='bold')
ax.set_title('Validación Random Forest — Reserva Pereyra Iraola', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(OUTPUT_DIR + 'Scatter_Validacion_RF.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Scatter plot guardado.')


In [ ]:
# ============================================================
# MÓDULO 6 — Mapa de carbono aéreo predicho
# ============================================================

with rasterio.open(predicho_tif) as src:
    carbono_map = src.read(1)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(
    carbono_map,
    cmap='RdYlGn',
    vmin=np.nanpercentile(carbono_map, 2),
    vmax=np.nanpercentile(carbono_map, 98)
)
cbar = plt.colorbar(im, ax=ax, label='Carbono aéreo (tC/ha)', pad=0.02)

stats_text = (f'Media: {np.nanmean(carbono_map):.2f} tC/ha\n'
              f'SD: {np.nanstd(carbono_map):.2f}\n'
              f'Mín: {np.nanmin(carbono_map):.2f}\n'
              f'Máx: {np.nanmax(carbono_map):.2f}')
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_title('Estimación de carbono aéreo — Reserva de Biósfera Pereyra Iraola',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Coordenada X (píxeles)', fontsize=11)
ax.set_ylabel('Coordenada Y (píxeles)', fontsize=11)
plt.tight_layout()

plt.savefig(OUTPUT_DIR + 'Mapa_Carbono_RF.png', dpi=300, bbox_inches='tight')
plt.show()

n_pixeles_validos = np.sum(~np.isnan(carbono_map))
stock_total = np.nansum(carbono_map) * 0.01   # 10m × 10m = 0.01 ha/píxel
print(f'✅ Mapa guardado.')
print(f'   Stock total estimado: {stock_total/1000:.2f} ktoneladas C')


In [ ]:
# ============================================================
# MÓDULO 6 — Mapa de residuos (observado − predicho)
# ============================================================
from matplotlib.colors import TwoSlopeNorm

with rasterio.open(carbono_10m_tif) as src:
    c_obs = src.read(1).astype(float)
with rasterio.open(predicho_tif) as src:
    c_pred = src.read(1).astype(float)
    perfil = src.profile

mask_r = (~np.isnan(c_obs)) & (~np.isnan(c_pred)) & (c_obs > 0)
residuos = np.full_like(c_obs, np.nan)
residuos[mask_r] = c_obs[mask_r] - c_pred[mask_r]

# Guardar GeoTIFF de residuos
residuos_tif = OUTPUT_DIR + 'Carbono_Residuos_RF.tif'
perfil.update(dtype='float32', count=1)
with rasterio.open(residuos_tif, 'w', **perfil) as dst:
    dst.write(residuos.astype('float32'), 1)

# Visualizar
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im0 = axes[0].imshow(residuos, cmap='RdYlBu_r', norm=norm)
plt.colorbar(im0, ax=axes[0], label='Error (tC/ha)')
axes[0].set_title('Mapa de residuos (Observado − Predicho)')
axes[0].axis('off')

axes[1].hist(residuos[mask_r].ravel(), bins=60, color='gray', edgecolor='black', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', label='Error cero')
axes[1].axvline(np.nanmean(residuos), color='blue', linestyle='-',
                label=f'Bias = {np.nanmean(residuos):.2f} tC/ha')
axes[1].set_xlabel('Error (tC/ha)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del error')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'Mapa_Residuos_RF.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Residuos guardados: {residuos_tif}')
print(f'   Bias medio: {np.nanmean(residuos):.2f} tC/ha')


## Módulo 7 — Análisis estratificado del error por clase de altura

In [ ]:
# ============================================================
# MÓDULO 7 — Error estratificado por clase de altura del dosel
# ============================================================

altura_class_tif = OUTPUT_DIR + 'Altura_Dosel_Pereyra_Class.tif'
residuos_tif     = OUTPUT_DIR + 'Carbono_Residuos_RF.tif'

with rasterio.open(residuos_tif) as src_r:
    residuos = src_r.read(1).astype(float)

with rasterio.open(altura_class_tif) as src_c:
    if src_c.shape != residuos.shape:
        clases = src_c.read(
            1,
            out_shape=(1, residuos.shape[0], residuos.shape[1]),
            resampling=Resampling.nearest
        ).astype(int)
    else:
        clases = src_c.read(1).astype(int)

# Etiquetas de clase de altura
class_labels = {1: '< 1 m', 2: '1–5 m', 3: '5–10 m',
                4: '10–15 m', 5: '15–20 m', 6: '> 20 m'}

stats = []
for cv in sorted(np.unique(clases[~np.isnan(clases)])):
    m = (clases == cv) & (~np.isnan(residuos))
    r = residuos[m]
    if r.size > 0:
        stats.append({
            'Clase': class_labels.get(int(cv), str(cv)),
            'N muestras': len(r),
            'Error medio (tC/ha)': round(np.mean(r), 2),
            'Desv. estándar': round(np.std(r), 2),
            'RMSE aprox.': round(np.sqrt(np.mean(r)**2 + np.std(r)**2), 2),
        })

df_stats = pd.DataFrame(stats)
print('📊 Error estratificado por clase de altura del dosel')
print(df_stats.to_string(index=False))

# Guardar CSV
df_stats.to_csv(OUTPUT_DIR + 'Error_Por_Clase_Altura.csv', index=False)
print('\n✅ Tabla guardada: Error_Por_Clase_Altura.csv')


## Módulo 8 — Importancia de variables espectrales

In [ ]:
# ============================================================
# MÓDULO 8 — Feature importance del modelo Random Forest
# ============================================================
# Resultado publicado en el paper: B11 (SWIR1) = 59.1%

feature_names = ['B2','B3','B4','B8','B11','B12','NDVI','NDWI']
importances   = rf.feature_importances_

df_imp = pd.DataFrame({
    'Variable': feature_names,
    'Importancia': importances
}).sort_values('Importancia', ascending=False)

print('📈 Importancia de variables (Feature Importance — Gini):')
print(df_imp.to_string(index=False))

# Gráfico horizontal
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2ecc71' if v == 'B11' else '#3498db' for v in df_imp['Variable']]
ax.barh(df_imp['Variable'], df_imp['Importancia'],
        color=colors, edgecolor='darkslategray', linewidth=1.2)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('Importancia relativa (Gini)', fontsize=12)
ax.set_ylabel('Variable espectral', fontsize=12)
ax.set_title('Importancia de variables — Random Forest\nReserva de Biósfera Pereyra Iraola',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.5)

# Anotar valores
for i, (_, row) in enumerate(df_imp.iterrows()):
    ax.text(row['Importancia'] + 0.005, i, f"{row['Importancia']:.3f}",
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'Importancia_Variables_RF.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado.')


## Verificación final — archivos generados

In [ ]:
# ============================================================
# VERIFICACIÓN FINAL — archivos generados por el pipeline
# ============================================================

archivos_esperados = [
    'GEDI_RH95_Pereyra.tif',
    'Sentinel2_Pereyra_Stack.tif',
    'Carbono_Pereyra.tif',
    'Carbono_Pereyra_10m.tif',
    'Carbono_Predicho_RF.tif',
    'Carbono_Residuos_RF.tif',
    'Scatter_Validacion_RF.png',
    'Mapa_Carbono_RF.png',
    'Mapa_Residuos_RF.png',
    'Importancia_Variables_RF.png',
    'Error_Por_Clase_Altura.csv',
]

print(f'Verificando archivos en: {OUTPUT_DIR}\n')
todos_ok = True
for f in archivos_esperados:
    path = OUTPUT_DIR + f
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {f:<45} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ FALTANTE: {f}')
        todos_ok = False

print()
if todos_ok:
    print('🎉 Pipeline completo. Todos los archivos generados correctamente.')
else:
    print('⚠️  Algunos archivos faltan. Revisá los módulos anteriores.')


## Referencias bibliográficas

- Belgiu, M. y Drăguț, L. (2016). Random forest in remote sensing. *ISPRS Journal of Photogrammetry and Remote Sensing, 114*, 24–31. https://doi.org/10.1016/j.isprsjprs.2016.01.011
- Chave, J., et al. (2014). Improved allometric models to estimate the aboveground biomass of tropical trees. *Global Change Biology, 20*(10), 3177–3190. https://doi.org/10.1111/gcb.12629
- Dubayah, R., et al. (2020). The Global Ecosystem Dynamics Investigation. *Science of Remote Sensing, 1*, 100002. https://doi.org/10.1016/j.srs.2020.100002
- Dubayah, R., et al. (2021). GEDI L2A Elevation and Height Metrics Data Global Footprint Level V002. NASA EOSDIS. https://doi.org/10.5067/GEDI/GEDI02_A.002
- Lei, Y., et al. (2024). Estimating forest canopy height based on GEDI LiDAR data. *ISPRS Archives, XLVIII-1*, 297–303. https://doi.org/10.5194/isprs-archives-XLVIII-1-2024-297-2024
- Li, X., et al. (2022). Integrating Sentinel-2 imagery and ML to estimate aboveground biomass. *Remote Sensing, 14*(8), 1913. https://doi.org/10.3390/rs14081913
- Saatchi, S. S. y Favrichon, S. (2023). Global Vegetation Height Metrics from GEDI. ORNL DAAC. https://doi.org/10.3334/ORNLDAAC/2294
- Sione, S. M., et al. (2021). Estimación del contenido y captura potencial de carbono en el Espinal. *FAVE Sección Ciencias Agrarias, 20*(1), 331–349.
- Winrock International. (2018). *Calculating carbon stocks: A guidance module*.

---

*Paper completo disponible en: CAI 2026 — 55 JAIIO, La Plata, Argentina*
